# Week 2 Exercise - Agentic Sales Email System

Build a Cold Sales Outreach Email Agent System using the OpenAI Agents SDK.

**Parts:**
1. Agent Workflow - 3 specialized sales agents + a picker agent
2. Tools & Agent Interactions - `@function_tool`, `.as_tool()`, email sending
3. Agent Collaboration & Handoffs - Sales Manager orchestration with handoffs

---

## Setup

Make sure you have a `.env` file with:
```
OPENAI_API_KEY=your_key_here
SENDGRID_API_KEY=your_key_here  # Optional - will fallback to console output
```

If you want to use SendGrid:
1. Sign up at https://sendgrid.com/
2. Create an API key under Settings >> API Keys
3. Verify your sender email under Settings >> Sender Authentication

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import asyncio

load_dotenv(override=True)

# Optional: SendGrid imports
try:
    import sendgrid
    from sendgrid.helpers.mail import Mail, Email, To, Content
    HAS_SENDGRID = bool(os.environ.get('SENDGRID_API_KEY'))
except ImportError:
    HAS_SENDGRID = False

print(f"SendGrid available: {HAS_SENDGRID}")

### Email configuration

Update these with your own email addresses if using SendGrid.

In [ ]:
# Change these to your own email addresses
FROM_EMAIL = "your_verified_sender@example.com"
TO_EMAIL = "your_recipient@example.com"

---
## Part 1: Agent Workflow

We'll create 3 specialized sales agents with different writing styles, run them in parallel, then use a picker agent to select the best email.

### Step 1.1 - Define agent instructions

Each agent works for **ComplAI**, a SaaS tool for SOC2 compliance and audit preparation, powered by AI.

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### Step 1.2 - Create the 3 sales agents

In [ ]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

### Step 1.3 - Test a single agent with streaming

In [ ]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

### Step 1.4 - Run all 3 agents in parallel

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for i, output in enumerate(outputs, 1):
    print(f"=== Agent {i} ===")
    print(output)
    print()

### Step 1.5 - Create a picker agent to select the best email

In [ ]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

### Step 1.6 - Full pipeline: generate emails in parallel, then pick the best

In [ ]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

> **Check your trace at:** https://platform.openai.com/traces

---
## Part 2: Tools & Agent Interactions

Now we add tools using:
- `@function_tool` decorator to convert functions into tools
- `.as_tool()` to convert agents into tools

No more JSON boilerplate!

### Step 2.1 - Create the `send_email` tool

Uses SendGrid if available, otherwise prints to console.

In [ ]:
@function_tool
def send_email(body: str):
    """Send out an email with the given body to all sales prospects"""
    if HAS_SENDGRID:
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
        from_email = Email(FROM_EMAIL)
        to_email = To(TO_EMAIL)
        content = Content("text/plain", body)
        mail = Mail(from_email, to_email, "Sales email", content).get()
        sg.client.mail.send.post(request_body=mail)
        print(f"Email sent via SendGrid to {TO_EMAIL}")
    else:
        print("[SendGrid not configured - printing email instead]")
        print(f"TO: {TO_EMAIL}")
        print(f"FROM: {FROM_EMAIL}")
        print(f"SUBJECT: Sales email")
        print(f"BODY:\n{body}")
    return {"status": "success"}

In [ ]:
# Inspect the auto-generated tool
send_email

### Step 2.2 - Convert agents into tools with `.as_tool()`

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]
tools

### Step 2.3 - Sales Manager agent (orchestrates via tools)

This is the planning agent - it uses the 3 sales agent tools to generate drafts, picks the best, and sends it.

In [ ]:
sales_manager_instructions_v1 = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts - do not write them yourself.
- You must send ONE email using the send_email tool - never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions_v1,
    tools=tools,
    model="gpt-4o-mini"
)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

print(result.final_output)

> **Check your trace** at https://platform.openai.com/traces and check your email (or console output above)!

---
## Part 3: Agent Collaboration & Handoffs

Handoffs pass **control** from one agent to another (unlike tools, where control returns to the caller).

We'll add:
- **Subject Writer** agent - writes email subjects
- **HTML Converter** agent - converts text to HTML email
- **Email Manager** agent - orchestrates formatting and sending via handoff

### Step 3.1 - Create the Subject Writer and HTML Converter agents

In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

### Step 3.2 - Create the `send_html_email` tool

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects"""
    if HAS_SENDGRID:
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
        from_email = Email(FROM_EMAIL)
        to_email = To(TO_EMAIL)
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content).get()
        sg.client.mail.send.post(request_body=mail)
        print(f"HTML email sent via SendGrid to {TO_EMAIL}")
    else:
        print("[SendGrid not configured - printing email instead]")
        print(f"TO: {TO_EMAIL}")
        print(f"FROM: {FROM_EMAIL}")
        print(f"SUBJECT: {subject}")
        print(f"HTML BODY:\n{html_body}")
    return {"status": "success"}

### Step 3.3 - Create the Email Manager agent (receives handoffs)

In [ ]:
emailer_tools = [subject_tool, html_tool, send_html_email]

emailer_instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=emailer_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it"
)

### Step 3.4 - Updated Sales Manager with handoffs

Now the Sales Manager uses **tools** for generating drafts and **handoff** for sending. This is the key difference:
- Tools: control returns to the Sales Manager
- Handoff: control passes to the Email Manager

In [ ]:
# Sales agent tools (no send_email - that's handled by the Email Manager now)
sales_tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

print("Tools:", sales_tools)
print("Handoffs:", handoffs)

In [ ]:
sales_manager_instructions_v2 = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts - do not write them yourself.
- You must hand off exactly ONE email to the Email Manager - never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions_v2,
    tools=sales_tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

print(result.final_output)

> **Check your trace** at https://platform.openai.com/traces and check your email!

---
## Exercise Challenges

### Challenge 1: Identify the patterns
Can you identify the **Agentic design patterns** used in this notebook?
- Parallelization
- Orchestrator-workers
- Tool use
- Evaluator-optimizer

### Challenge 2: Workflow vs Agent
What is the **1 line** that changed this from being an Agentic "workflow" to an "agent" under Anthropic's definition?

**Hint:** It's the addition of `handoffs` - when an agent can autonomously decide to delegate to another agent, it crosses from a deterministic workflow to an agent with autonomous decision-making.

### Challenge 3: Extend the system
Try adding more tools and agents:
- A **mail merge** tool that sends to a list of recipients
- A **CRM lookup** tool that retrieves prospect information
- A **follow-up agent** that writes follow-up emails based on the original

### Challenge 4 (HARD): Webhook-based reply handling
Research how to have SendGrid call a callback webhook when a user replies to an email, then have the SDR respond to keep the conversation going!

---
## Scratch space for challenges

Use the cells below to work on the challenges.

In [ ]:
# Challenge 3: Add your extended tools and agents here


In [ ]:
# Challenge 4: Webhook-based reply handling
